[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc3_stats/exercices/seance3_exercices.ipynb)

# Séance 3.3 — Relier deux variables — y a-t-il un lien ?

**Exercices** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- réutiliser `pd.cut` (vu en séance 2.2) pour fabriquer une variable qualitative
- tester le lien entre deux variables qualitatives avec un khi-deux
- lire un tableau d'effectifs attendus pour dire *où* est la dépendance
- mesurer un lien entre deux variables quantitatives (Pearson, Spearman)
- reconnaître les trois pièges de la corrélation : extrêmes, non-linéarité, variable de confusion

## Comment ça marche

La feuille compte **deux parties**, à faire dans l'ordre.

**Partie 1 — l'échauffement.** Le code est déjà écrit, il ne reste que les `____` à
remplir. Chaque exercice se termine par une cellule de **vérification** qui vous dit
immédiatement si votre réponse est bonne.

**Partie 2 — les questions.** Une question, une cellule **vide** : à vous d'écrire le
code entier. Il n'y a pas de vérification automatique — on les corrige ensemble en
séance, et la correction est publiée après.

> ⚠️ Si une vérification de la partie 1 affiche `NameError`, c'est que la cellule
au-dessus n'a pas été exécutée, ou qu'il y reste un `____`. Complétez-la, relancez-la,
puis relancez la vérification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc3_stats/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")

# pd.cut decoupe une colonne continue en categories ; 1e9 = "et au-dela"
cmd["taille"] = pd.cut(cmd["ca"], [0, 200, 500, 1e9],
                       labels=["petite", "moyenne", "grande"])
print(cmd["taille"].value_counts())

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Le tableau croisé

> **Votre mission :**
> - Croiser `pays` (en lignes) et `taille` (en colonnes) pour les quatre pays les plus présents → `tab`.
> - Combien de grosses commandes irlandaises ? → `irl_grande`

In [ ]:
top4 = cmd["pays"].value_counts().head(4).index
sub = cmd.query("pays in @top4")

tab = pd.crosstab(sub["____"], sub["taille"])
irl_grande = tab.loc["Irlande", "grande"]
print(tab)

In [ ]:
verifier("1 - grosses commandes irlandaises", irl_grande == 157,
         "pd.crosstab(lignes, colonnes) puis .loc['Irlande', 'grande']")

### Exercice 2 — Le khi-deux

> **Votre mission :**
> - Tester le lien entre pays et taille de commande sur `tab`.
> - Récupérer la p-value → `p_khi2`, et conclure au seuil de 5 % → `dependant` (`True`/`False`).
> - Rappel : les quatre noms à gauche du `=` se lisent **dans l'ordre**.

In [ ]:
khi2, p_khi2, ddl, attendus = stats.chi2_contingency(____)

dependant = p_khi2 < 0.05
print("p =", p_khi2, "| dependance :", dependant)

In [ ]:
verifier("2 - dependance pays / taille", bool(dependant) and p_khi2 < 1e-20,
         "passez le tableau croise a stats.chi2_contingency")

### Exercice 3 — Où est la dépendance ?

> **Votre mission :**
> - Construire le tableau des écarts (observé − attendu) → `ecarts`.
> - Quel est l'écart des grosses commandes britanniques ? → `ecart_uk` (arrondi à 1 décimale)

In [ ]:
att = pd.DataFrame(attendus, index=tab.index, columns=tab.columns)
ecarts = tab - ____

ecart_uk = round(ecarts.loc["Royaume-Uni", "grande"], 1)
print(ecarts.round(1))

In [ ]:
verifier("3 - ecart des grosses commandes britanniques", ecart_uk == -82.7,
         "observe moins attendu, donc tab - att")

### Exercice 4 — Les corrélations d'un coup

> **Votre mission :**
> - Afficher la matrice de corrélation de `ca`, `nart` et `qte`.
> - En extraire la corrélation entre `ca` et `qte` → `r_ca_qte` (arrondie à 3 décimales).

In [ ]:
print(cmd[["ca", "nart", "qte"]].corr().round(3))

r_ca_qte = round(cmd["ca"].corr(cmd["____"]), 3)
print(r_ca_qte)

In [ ]:
verifier("4 - correlation ca / qte", r_ca_qte == 0.848,
         "df['ca'].corr(df['qte'])")

### Exercice 5 — Pearson contre Spearman

> **Votre mission :**
> - Calculer les deux corrélations entre `ca` et `nart` → `r_pearson` et `r_spear` (arrondies à 3 décimales).
> - Laquelle est la plus élevée, et pourquoi ?

In [ ]:
r_pearson = round(cmd["ca"].corr(cmd["nart"]), 3)
r_spear = round(cmd["ca"].corr(cmd["nart"], method="____"), 3)

print("Pearson", r_pearson, "| Spearman", r_spear)

In [ ]:
verifier("5a - Pearson", r_pearson == 0.382, "c'est la methode par defaut")
verifier("5b - Spearman", r_spear == 0.671, "method='spearman'")

### Exercice 6 — Le poids de 1 % des lignes

> **Votre mission :**
> - Retirer les 1 % de commandes les plus grosses → `sans`.
> - Recalculer la corrélation entre `ca` et `nart` sur ce sous-ensemble → `r_sans` (arrondie à 3 décimales).

In [ ]:
seuil = cmd["ca"].quantile(____)
sans = cmd.query("ca < @seuil")

r_sans = round(sans["ca"].corr(sans["nart"]), 3)
print(len(cmd) - len(sans), "lignes retirees | correlation :", r_sans)

In [ ]:
verifier("6 - correlation sans les extremes", r_sans == 0.489,
         "les 1 % du haut commencent au quantile 0.99")

### Exercice 7 — Le même chiffre, deux pays

> **Votre mission :**
> - Calculer la corrélation `ca` / `nart` **en Belgique** → `r_be`, puis **en Irlande** → `r_irl`.
> - Arrondir à 3 décimales. Comparez au 0,382 global.

In [ ]:
be = cmd.query("pays == 'Belgique'")
irl = cmd.query("pays == '____'")

r_be = round(be["ca"].corr(be["nart"]), 3)
r_irl = round(irl["ca"].corr(irl["nart"]), 3)
print("Belgique", r_be, "| Irlande", r_irl)

In [ ]:
verifier("7a - correlation belge", r_be == 0.903, "filtrez d'abord, correlez ensuite")
verifier("7b - correlation irlandaise", r_irl == 0.234,
         "le nom du pays s'ecrit Irlande, avec une majuscule")

### Exercice 8 — Question de synthèse

> **Votre mission :**
> - On vous demande : *« le nombre de produits distincts commandés explique-t-il le montant ? »*
> - Calculer la corrélation de Spearman `ca` / `nart` sur le seul Royaume-Uni → `r_uk` (3 décimales).
> - Puis tracer le nuage de points correspondant.
> - Enfin, écrivez votre réponse en commentaire — en trois phrases maximum.

In [ ]:
uk = cmd.query("pays == 'Royaume-Uni'")
r_uk = round(uk["ca"].corr(uk["nart"], method="spearman"), 3)

uk.plot(kind="scatter", x="nart", y="ca", alpha=0.3, figsize=(7, 4))
plt.xlabel("nart : nombre de produits distincts")
plt.ylabel("ca : montant de la commande (euros)")
plt.show()
print("Spearman au Royaume-Uni :", r_uk)

# Votre reponse :

In [ ]:
verifier("8 - Spearman au Royaume-Uni", r_uk == 0.574,
         "filtrez sur le Royaume-Uni, puis method='spearman'")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 9 — Une dépendance certaine, mais forte ?

> **Votre mission :**
> - Le khi-deux dit qu'une dépendance **existe**. Il ne dit rien de sa **force** — exactement comme la p-value du test t.
> - Calculer le **V de Cramér** : `sqrt(khi2 / (n * (k - 1)))`, où `n` est l'effectif total du tab et `k` le plus petit de ses deux nombres de modalités.
> - Il vaut 0 quand il n'y a aucun lien, 1 quand la connaissance de l'un détermine l'autre. Que vaut-il ici ?

### Question 10 — Quelles cases sont vraiment anormales ?

> **Votre mission :**
> - Le tab des écarts brut (observé − attendu) dépend des effectifs : un écart de 20 est énorme sur une case attendue à 30, négligeable sur une case attendue à 3 000.
> - Calculer les **résidus standardisés** : `(observé - attendu) / sqrt(attendu)`.
> - Repère usuel : au-delà de **2 en valeur absolue**, la case sort de l'ordinaire. Lesquelles ?

### Question 11 — Quand le khi-deux n'est pas applicable

> **Votre mission :**
> - Croiser `jour` et `pays` sur **tous** les pays, puis compter les effectifs **attendus** inférieurs à 5.
> - Recommencer sur les quatre pays les plus présents.
> - Le khi-deux exige des effectifs attendus d'au moins 5. Laquelle des deux p-values a le droit d'être citée ?

### Question 12 — Pourquoi Pearson sous-estimait

> **Votre mission :**
> - En partie 1 : Pearson 0,38 contre Spearman 0,67 pour `ca` et `nart`, parce que la relation est **courbe**.
> - Recalculer la corrélation de Pearson sur les **logarithmes** des deux colonnes.
> - Comparer les trois nombres. Qu'est-ce que le logarithme a fait à la relation ?
> - *Nouveau :* `np.log(serie)`.

### Question 13 — La corrélation dépend du niveau d'observation

> **Votre mission :**
> - Calculer la corrélation `ca` / `nart` à trois niveaux : par **commande**, puis en **totalisant** le CA et le nombre de produits par **client**, puis par **pays**.
> - Trois nombres très différents à partir des mêmes données.
> - Lequel citeriez-vous dans une note, et pourquoi les deux autres seraient-ils trompeurs ?

### Question 14 — Conclure proprement à l'absence de lien

> **Votre mission :**
> - Le jour de la semaine influence-t-il la taille des commandes ? Croiser `jour` et `taille`, puis tester (vous devriez trouver p = 0,7866).
> - Ajouter les deux éléments qui manquent pour pouvoir écrire quelque chose : le **V de Cramér**, et le tab des pourcentages par ligne.
> - Rédigez la phrase de conclusion en commentaire. Attention à ne pas écrire « il n'y a pas de lien ».

### Question 15 — Question de synthèse

> **Votre mission :**
> - On vous demande : *« le pays détermine-t-il la façon d'acheter ? »*
> - Répondez avec trois éléments : l'existence du lien (p), sa force (V de Cramér), et l'endroit où il se situe (les résidus).
> - Puis, en commentaire, la conclusion en trois phrases — dont une qui dit ce que ces données **ne** permettent **pas** d'affirmer.